In [ ]:
 !pip install sympy gradio -q

import sympy as sp
import gradio as gr
from sympy import symbols, sympify, solve, Eq, simplify, diff, integrate, latex

# Create symbol
x = symbols('x')

def clean_math_expression(text):
    text = text.lower().strip()

    if 'solve' in text:
        math_part = text.replace('solve', '').strip()
    elif 'simplify' in text:
        math_part = text.replace('simplify', '').strip()
    elif 'derivative' in text or 'diff' in text:
        math_part = text.replace('derivative', '').replace('diff', '').strip()
    elif 'integrate' in text:
        math_part = text.replace('integrate', '').strip()
    else:
        math_part = text

    math_part = ' '.join(math_part.split())
    return math_part


def parse_math_problem(user_input):

    if not user_input:
        return "Error", None, "Please enter a math problem"

    math_expr = clean_math_expression(user_input)

    try:

        # Equation solving
        if '=' in math_expr:

            left, right = math_expr.split('=', 1)

            left_expr = sympify(left.strip())
            right_expr = sympify(right.strip())

            equation = Eq(left_expr, right_expr)

            solution = solve(equation, x)

            return "Solution", solution, f"Solving equation: {left.strip()} = {right.strip()}\nSolutions: {solution}"

        # Simplify
        elif 'simplify' in user_input.lower():

            expr = sympify(math_expr)
            result = simplify(expr)

            return "Simplified", result, f"Simplified result: {result}"

        # Derivative
        elif any(word in user_input.lower() for word in ['derivative','diff']):

            expr = sympify(math_expr)
            result = diff(expr, x)

            return "Derivative", result, f"d/dx({expr}) = {result}"

        # Integral
        elif 'integrate' in user_input.lower():

            expr = sympify(math_expr)
            result = integrate(expr, x)

            return "Integral", result, f"∫ {expr} dx = {result}"

        # Default simplify
        else:

            expr = sympify(math_expr)
            result = simplify(expr)

            return "Simplified", result, f"Result: {result}"

    except Exception as e:

        return "Error", None, f"Could not parse: '{math_expr}'\n\nError: {str(e)[:100]}\n\nTry formats like:\nsolve x**2 + 2*x + 1 = 0\n derivative x**3\n integrate sin(x)\n simplify (x+1)**2"



def math_tutor_interface(user_input):

    if not user_input:
        return "Enter a math problem."

    result_type, result, explanation = parse_math_problem(user_input)

    if result_type == "Error":
        return f"❌ {explanation}"

    try:
        latex_result = f"${latex(result)}$"
    except:
        latex_result = str(result)

    output = f"""
### ✅ {result_type}

**Final Answer**

{latex_result}

**Explanation**

{explanation}
"""

    return output


# PROFESSIONAL UI

with gr.Blocks(theme=gr.themes.Soft()) as demo:

    gr.Markdown("# 🧠 AI Math Tutor")

    gr.Markdown(
        "Solve **equations, derivatives, integrals and simplifications**.\n\n"
        "**Use these formats:**\n"
        "- `solve x**2 + 2*x + 1 = 0`\n"
        "- `derivative x**3`\n"
        "- `integrate sin(x)`\n"
        "- `simplify (x+1)**2`\n\n"
        "Use `**` for powers (example: x**2)"
    )

    with gr.Row():

        with gr.Column():

            user_input = gr.Textbox(
                label="Enter Math Problem",
                placeholder="Example: solve x**2 + 2*x + 1 = 0",
                lines=3
            )

            solve_btn = gr.Button("🚀 Solve It", variant="primary")

            clear_btn = gr.Button("Clear")

        with gr.Column():

            output = gr.Markdown(label="Solution")

    gr.Markdown("### Example Problems")

    gr.Examples(
        examples=[
            "solve x**2 + 2*x + 1 = 0",
            "derivative x**3",
            "integrate sin(x)",
            "simplify (x+1)**2"
        ],
        inputs=user_input
    )

    solve_btn.click(
        fn=math_tutor_interface,
        inputs=user_input,
        outputs=output
    )

    clear_btn.click(
        lambda: "",
        None,
        user_input
    )


print("AI Math Tutor ready!")
demo.launch(share=True)

/tmp/ipykernel_438/3536583594.py:121: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


AI Math Tutor ready!
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://c5a316e52dba119b79.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
